# 09 · Deep model on the FPGA: QAT, deployable accuracy, HLS export
Notebook 08 found that **depth** is what helps (fused R² 0.688 → 0.757 with 2 layers, 0.783 with 3, every subject better),
but those models were trained in float. This notebook turns them into something the ZedBoard can run, and settles one open question.

| step | what | why |
|---|---|---|
| 1 | retrain `base` and `L2` with early-stopping patience 12 (`base_p12`, `L2_p12`) | in notebook 08 the 1-layer model stopped exactly when the learning rate was first halved; same seed, so these runs retrace the originals and only train longer. Tells whether part of the depth gain was the schedule |
| 2 | `L2_ptq10`: L2 quantised **without** retraining, 10-bit activations | how much fixed point costs before QAT (the float L2 has LayerNorm outputs up to 7.9 against a ±4 range) |
| 3 | QAT fine-tunes from the notebook 08 checkpoints | |
|   | `L2_q10`: 2 layers, activations (10,3), weights (10,3) = current hardware formats | 2.9 Mbit of weights, fits |
|   | `L2_q12`: 2 layers, activations (12,4), weights (10,3) | wider activation range (±8) for the deeper layers |
|   | `L3_w8`: 3 layers, activations (12,4), weights **(8,2)** | 3.4 Mbit: just inside the weight budget |
| 4 | sequential evaluation of all of them (own estimates, Kalman re-tuned, per-subject bootstrap) | QAT models run with fake quantisation at the HLS points, so their numbers are the **deployable** ones |
| 5 | export the best QAT model that fits on chip | weights, model_config.h, golden data, HIL windows, firmware constants incl. the fusion gain |

**Before running**
* Re-upload the `emg_common.py` utility script (multi-layer export, `kalman_gain`, evaluation about 20x faster).
* Attach: NinaPro DB2, **notebook 07's output** (preprocessed 200 ms data and the `base` checkpoint) and **notebook 08's output** (`scale_L2.pt`, `scale_L3.pt`).
* Accelerator: GPU. The first cell must print `cuda True`.
* Expected time about 3 hours. Resumable: attach this notebook's own output and re-run; finished training and evaluations are reused.
* Click **Save Version** at the end and download `qat/` (it contains `hls_export_<model>.zip`).

Environment variables (optional): `EMG_QAT_EPOCHS` (default 8), `EMG_SKIP_SCHED=1` skips step 1, `EMG_EXPORT=<name>` forces the exported model.

In [ ]:
import os, sys, glob, json, time, copy, shutil
for p in ['/kaggle/usr/lib/notebooks/quddusikashaf/emg_common', '/kaggle/usr/lib/emg_common', '/kaggle/usr/lib/emg-common', '.', '..', 'kaggle_scripts']:
    if os.path.exists(os.path.join(p, 'emg_common.py')):
        sys.path.insert(0, os.path.abspath(p)); break
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import emg_common as ec
assert hasattr(ec, 'kalman_gain') and hasattr(ec, 'hls_unsupported'), 're-upload the updated emg_common.py utility script'
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)
WORK = os.environ.get('EMG_WORK', '/kaggle/working')
N_BOOT = int(os.environ.get('EMG_NBOOT', 2000)); QAT_EPOCHS = int(os.environ.get('EMG_QAT_EPOCHS', 8))
OUT = f'{WORK}/qat'; os.makedirs(f'{OUT}/outputs', exist_ok=True)
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| emg_common', ec.__file__)
if not torch.cuda.is_available() and 'EMG_WORK' not in os.environ:
    print('WARNING: no GPU. Settings > Accelerator > GPU T4, then restart; on CPU this notebook takes over a day.')

## 1 · Data: 200 ms windows, 50 ms step (reused from notebook 07)

In [ ]:
WIN, STEP = 400, 100
TAG = f'db2_processed_w{WIN}_s{STEP}'
BASE = os.environ.get('EMG_BASE_PATH')
if BASE is None:
    hits = sorted(glob.glob('/kaggle/input/**/DB2_s1/DB2_s1', recursive=True))
    BASE = os.path.dirname(os.path.dirname(hits[0])) if hits else ec.Config.base_path
dcfg = ec.Config(base_path=BASE, processed_dir=f'{WORK}/{TAG}', seq_len=WIN, stride=STEP,
                 n_subjects=int(os.environ.get('EMG_N_SUBJECTS', 10)), causal_filter=True, train_reps=(1, 3, 4, 6), test_reps=(2, 5))
reuse = sorted(glob.glob(f'/kaggle/input/**/{TAG}', recursive=True))
subjects = ec.list_subjects(dcfg) if not reuse else sorted(os.path.basename(os.path.dirname(f)) for f in glob.glob(f'{reuse[0]}/*/data.npz'))
npz_files = []
for s in subjects[:dcfg.n_subjects]:
    found = [f'{d}/{s}/data.npz' for d in [dcfg.processed_dir] + reuse if os.path.exists(f'{d}/{s}/data.npz')]
    npz_files.append(found[0] if found else ec.process_subject(dcfg, s))
print(len(npz_files), 'subjects from', os.path.dirname(os.path.dirname(npz_files[0])))

## 2 · Models (resumable)
Float parents come from notebooks 07/08. Schedule-check runs copy their parent's configuration and seed and change only the patience
(and allow up to 60 epochs). QAT runs start from the float weights (`train_model(..., init_model=...)` with `epochs=0`), fake-quantise
weights and activations at exactly the HLS quantisation points, and keep the best epoch on the validation absolute-head error.

In [ ]:
device = ec.get_device()
PARENTS = {'base': 'w200ms_patch16_t25', 'L2': 'scale_L2', 'L3': 'scale_L3'}
SCHED = [] if os.environ.get('EMG_SKIP_SCHED') == '1' else [dict(name='base_p12', parent='base'), dict(name='L2_p12', parent='L2')]
QAT = [dict(name='L2_q10', parent='L2', act_fmt=(10, 3), weight_fmt=(10, 3)),
       dict(name='L2_q12', parent='L2', act_fmt=(12, 4), weight_fmt=(10, 3)),
       dict(name='L3_w8',  parent='L3', act_fmt=(12, 4), weight_fmt=(8, 2))]

def find_ckpt(name):
    hits = [f'{OUT}/outputs/{name}.pt'] + sorted(glob.glob(f'/kaggle/input/**/{name}.pt', recursive=True))
    return next((p for p in hits if os.path.exists(p)), None)

def clone_cfg(c, **kw):
    c2 = ec.Config.from_dict(c.to_dict())
    for k, v in kw.items(): setattr(c2, k, v)
    return c2

models, train_minutes = {}, {}
for name, ck in PARENTS.items():
    p = find_ckpt(ck); assert p, f'{ck}.pt not found: attach the output of notebook {"07" if name == "base" else "08"}'
    m, c, extra = ec.load_checkpoint(p, device); models[name] = (m.eval(), c)
    assert c.seq_len == WIN and c.patch_stride == 16 and not extra.get('qat'), (name, c.seq_len, c.patch_stride)
    print(f'{name:9s} <- {p}  layers {c.num_layers}  params {ec.count_params(m):,}  phase {extra.get("phase")}')

def train_or_reuse(name, c, init_model=None):
    p = find_ckpt(name)
    if p:
        m, c2, extra = ec.load_checkpoint(p, device); print(f'reusing {p} ({extra.get("phase")})'); return m.eval(), c2
    run = ec.init_wandb(c); t0 = time.time()
    m, info = ec.train_model(c, npz_files, run=run, tag=name, init_model=init_model)
    train_minutes[name] = (time.time() - t0) / 60; print(f'{name}: {train_minutes[name]:.1f} min')
    if run is not None: run.finish()
    m, c2, _ = ec.load_checkpoint(info['checkpoint'], device)      # best epoch, with the QAT formats attached
    return m.eval(), c2

for e in SCHED:
    pm, pc = models[e['parent']]
    c = clone_cfg(pc, patience=12, epochs=60, qat_epochs=0, save_dir=f'{OUT}/outputs', run_name=e['name'],
                  use_wandb=os.environ.get('EMG_WANDB', '1') == '1', project_name='ninapro_db2_qat')
    print(f"\n=== {e['name']}: {e['parent']} retrained, patience 12 (seed {c.seed})")
    models[e['name']] = train_or_reuse(e['name'], c)

for e in QAT:
    pm, pc = models[e['parent']]
    c = clone_cfg(pc, epochs=0, qat_epochs=QAT_EPOCHS, qat_lr=5e-5, patience=4, act_fmt=e['act_fmt'], weight_fmt=e['weight_fmt'],
                  save_dir=f'{OUT}/outputs', run_name=e['name'], use_wandb=os.environ.get('EMG_WANDB', '1') == '1', project_name='ninapro_db2_qat')
    cost = ec.hw_cost(c)
    print(f"\n=== {e['name']}: QAT from {e['parent']}, act {c.act_fmt}, weights {c.weight_fmt} | weights {cost['weights_mbit']:.2f} Mbit, "
          f"fits on chip: {cost['fits_on_chip']}, HLS: {cost['hls_supported']}")
    models[e['name']] = train_or_reuse(e['name'], c, init_model=pm)

# post-training quantisation of L2 (no retraining): what QAT has to recover
pm, pc = models['L2']; cq = clone_cfg(pc, act_fmt=(10, 3), weight_fmt=(10, 3))
models['L2_ptq10'] = (ec.convert_to_qat(copy.deepcopy(pm), cq).to(device).eval(), cq)
print('\nmodels:', list(models))

## 3 · Sequential evaluation (deployable: own estimates, filter re-tuned per model, per-subject bootstrap)
The filter is tuned per model over a finer range of gains than notebook 08 (whose best setting sat at the edge of its grid), so `base`
can come out slightly better here than in notebook 08; every model in this notebook is tuned the same way.
QAT and PTQ models run with fake quantisation, so their numbers include the fixed-point error of the HLS formats
(softmax, LayerNorm statistics and Q/K/V stay at higher precision, as in the HLS; the C simulation matches this to about 0.001).

In [ ]:
PARENT_OF = {e['name']: e['parent'] for e in QAT + SCHED}; PARENT_OF['L2_ptq10'] = 'L2'
done_csv, ps_csv = f'{OUT}/summary.csv', f'{OUT}/per_subject.csv'
rows = pd.read_csv(done_csv).to_dict(orient='records') if os.path.exists(done_csv) else []
per_subject = [pd.read_csv(ps_csv)] if os.path.exists(ps_csv) else []
done = {r['model'] for r in rows}
parse = lambda key: (float(key[0].split('=')[1]), float(key[1].split('=')[1]))
# The steady-state filter gain depends (almost) only on q/r. Notebook 08's grid picked its corner (q=1e-5, r=1e-3, gain 0.095),
# so sweep the ratio finely instead: gains from about 0.03 to 0.37.
RATIOS = (0.001, 0.002, 0.004, 0.007, 0.01, 0.015, 0.025, 0.04, 0.07, 0.12, 0.2)
QS, RS = (1e-5,), tuple(1e-5 / r for r in RATIOS)
print('candidate fusion gains:', [round(ec.kalman_gain(1e-5, r), 3) for r in RS])
for name, (m, c) in models.items():
    if name in done:
        print('already evaluated:', name); continue
    t0 = time.time()
    preds = [ec.predict_run(m, r, c, device, feedback=True) for f in npz_files for r in ec.ordered_runs(f, 'test') if len(r['angles']) >= 4]
    gk = ec.kalman_sweep(preds, qs=QS, rs=RS, use_delta=True, init='abs'); gs = ec.kalman_sweep(preds, qs=QS, rs=RS, use_delta=False, init='abs')
    kqr, sqr = parse(gk.stack().idxmin()), parse(gs.stack().idxmin())
    df = ec.per_subject_eval(m, npz_files, c, device, init='abs', kalman_qr=kqr, smooth_qr=sqr, n_boot=N_BOOT, tag=name)
    per_subject.append(df)
    a = df[df['subject'] == 'ALL'].iloc[0]; cost = ec.hw_cost(c, m); subj = df[df['subject'] != 'ALL']
    rows.append({'model': name, 'parent': PARENT_OF.get(name, ''), 'layers': c.num_layers, 'act_fmt': str(tuple(c.act_fmt)),
                 'weight_fmt': str(tuple(c.weight_fmt)), 'quantised': hasattr(m, 'act_fmt'), **cost, 'train_min': train_minutes.get(name),
                 'abs_only_r2': a['abs_only_r2'], 'smoothed_r2': a['abs_smoothed_r2'], 'fused_r2': a['kalman_r2'],
                 'fused_r2_lo': a['kalman_r2_lo'], 'fused_r2_hi': a['kalman_r2_hi'], 'fused_rmse': a['kalman_rmse'],
                 'subject_mean_r2': subj['kalman_r2'].mean(), 'subject_min_r2': subj['kalman_r2'].min(),
                 'delta_gain': a['delta_gain'], 'kalman_q': kqr[0], 'kalman_r': kqr[1], 'fusion_gain': ec.kalman_gain(*kqr)})
    print(f"{name}: fused R2 {a['kalman_r2']:.4f} [{a['kalman_r2_lo']:.4f}, {a['kalman_r2_hi']:.4f}] RMSE {a['kalman_rmse']:.4f} "
          f"| per-subject mean {subj['kalman_r2'].mean():.3f} | eval {time.time() - t0:.0f}s")
    pd.DataFrame(rows).to_csv(done_csv, index=False); pd.concat(per_subject, ignore_index=True).to_csv(ps_csv, index=False)
summary = pd.DataFrame(rows); per_subject = pd.concat(per_subject, ignore_index=True)
display(summary[['model', 'parent', 'layers', 'act_fmt', 'weight_fmt', 'quantised', 'weights_mbit', 'fits_on_chip', 'latency_ms_U8',
                 'fused_r2', 'fused_r2_lo', 'fused_r2_hi', 'fused_rmse', 'subject_mean_r2', 'subject_min_r2', 'delta_gain', 'fusion_gain']].round(4))

## 4 · Paired comparisons (per subject, bootstrap over subjects)
* **vs `base`**: is the deployable deep model still better than the 1-layer float baseline? Seed noise from notebook 08: 0.0023 RMSE.
* **vs its float parent**: what quantisation costs (QAT, PTQ) or what longer training buys (`*_p12`).

In [ ]:
SEED_NOISE = float(os.environ.get('EMG_SEED_NOISE', 0.0023))
rm = per_subject[per_subject['subject'] != 'ALL'].pivot(index='subject', columns='setting', values='kalman_rmse')
r2 = per_subject[per_subject['subject'] != 'ALL'].pivot(index='subject', columns='setting', values='kalman_r2')
rng = np.random.RandomState(0)
def paired(a, b):
    d = (rm[a] - rm[b]).values; boot = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(5000)]
    return {'model': a, 'reference': b, 'mean_rmse_change': d.mean(), 'ci_lo': np.percentile(boot, 2.5), 'ci_hi': np.percentile(boot, 97.5),
            'subjects_better': int((d < 0).sum()), 'mean_r2_change': (r2[a] - r2[b]).mean()}
comp = [paired(m, 'base') for m in rm.columns if m != 'base']
comp += [paired(m, PARENT_OF[m]) for m in rm.columns if m in PARENT_OF and PARENT_OF[m] in rm.columns]
comp = pd.DataFrame(comp)
comp['beyond_seed_noise'] = (comp['mean_rmse_change'].abs() > SEED_NOISE) & ((comp['ci_hi'] < 0) | (comp['ci_lo'] > 0))
comp.to_csv(f'{OUT}/paired.csv', index=False); display(comp.round(4))

fig, ax = plt.subplots(1, 2, figsize=(16, 5))
s = summary.sort_values('fused_r2')
ax[0].barh(s['model'], s['fused_r2'], xerr=[s['fused_r2'] - s['fused_r2_lo'], s['fused_r2_hi'] - s['fused_r2']], capsize=3,
           color=['tab:orange' if q else 'tab:blue' for q in s['quantised']])
ax[0].set_xlim(max(0, s['fused_r2_lo'].min() - 0.05), None); ax[0].set_xlabel('fused R² (pooled, 95 % CI); orange = fixed point (QAT/PTQ)')
ax[0].grid(alpha=.3)
vb = comp[comp['reference'] == 'base']
ax[1].barh(vb['model'], vb['mean_rmse_change'], xerr=[vb['mean_rmse_change'] - vb['ci_lo'], vb['ci_hi'] - vb['mean_rmse_change']], capsize=3)
ax[1].axvline(0, color='k', lw=1); ax[1].axvspan(-SEED_NOISE, SEED_NOISE, color='gray', alpha=.2, label='seed noise')
ax[1].set_xlabel('mean per-subject RMSE change vs base (negative = better)'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT}/qat_comparison.png', dpi=120); plt.show()

## 5 · Export the deployable model
Default choice: the quantised model with the lowest fused RMSE among those whose weights fit on chip and that the HLS supports
(`EMG_EXPORT=<name>` overrides). The export holds everything for `hls_code/scripts/install_export.py`; the firmware fusion gain
is the steady-state gain of the Kalman filter tuned above.

In [ ]:
cand = summary[summary['quantised'] & summary['fits_on_chip'] & summary['hls_supported'] & (summary['model'] != 'L2_ptq10')]
choice = os.environ.get('EMG_EXPORT') or cand.sort_values('fused_rmse').iloc[0]['model']
row = summary[summary['model'] == choice].iloc[0]; model, cfg = models[choice]
print(f"exporting {choice}: fused R2 {row['fused_r2']:.4f}, act {row['act_fmt']}, weights {row['weight_fmt']} ({row['weights_mbit']:.2f} Mbit), "
      f"fusion gain {row['fusion_gain']:.4f}")
EXPORT_DIR = f'{OUT}/hls_export_{choice}'
with np.load(npz_files[0]) as d:
    test_idx = np.where((d['split'] == 1) & (d['has_prev'] == 1))[0]
    pick = np.concatenate([[test_idx[len(test_idx) // 3]], test_idx[np.linspace(0, len(test_idx) - 1, int(os.environ.get('EMG_N_HIL', 40))).astype(int)]])
    X, restim, rep = d['sEMG'][pick], d['restim'][pick], d['rep'][pick]
sample_x = X[0].T.astype(np.float64)                       # [C, samples] channel-major = HLS input layout
hil = [(X[i].T.astype(np.float64), None, int(restim[i]), int(rep[i])) for i in range(1, len(pick))]
report = ec.export_hls(model, cfg, EXPORT_DIR, sample_x, None, hil, ec.load_norm_stats(npz_files[0]), fusion_gain=row['fusion_gain'])
display(pd.DataFrame(report).T.round(5))
print(open(f'{EXPORT_DIR}/model_config.h').read())
zip_path = shutil.make_archive(EXPORT_DIR, 'zip', EXPORT_DIR); print('archive:', zip_path)
json.dump({'exported': choice, 'summary': summary.to_dict(orient='records'), 'paired': comp.to_dict(orient='records')},
          open(f'{OUT}/qat_summary.json', 'w'), indent=2, default=str)
print(f"""
Install locally (repository root), then run the bit-accurate C simulation:
  unzip {os.path.basename(zip_path)} -d hls_export_{choice}
  python hls_code/scripts/install_export.py hls_export_{choice}
  cd hls_code && make clean && make run          # every layer must PASS against the fixed-point reference
Vitis: re-run C synthesis with the same hls_config.cfg (file list already updated); MAC_UNROLL=8 keeps latency near {row['latency_ms_U8']:.0f} ms.""")

### Reading the result
* **Schedule check**: if `base_p12` closes most of the gap to `L2`, the depth gain in notebook 08 was partly the training schedule.
  Compare `L2_p12` vs `base_p12` for the fair depth comparison.
* **Quantisation**: `L2_q10` / `L2_q12` vs `L2` (paired table, reference = parent) is the cost of fixed point after QAT; `L2_ptq10`
  shows the cost without QAT. A QAT model within seed noise of its float parent is a free deployment.
* **Choice**: the exported model is the deployable result. Quote its fused R² with the CI and the per-subject mean.
* Return to Claude: the `qat/` folder (summary.csv, per_subject.csv, paired.csv, qat_comparison.png, qat_summary.json,
  hls_export_<model>.zip) and the notebook log.